# Traffic Flow Prediction Using Road-Camera Footage
### Computer Vision & Deep Learning Course Project
**Author:** Mridul  
**Topic:** System that detects vehicles in road-camera footage using a CNN and uses an LSTM to predict short-term traffic flow.

---
## Project Stages Overview:
1. **Problem Definition**
2. **Dataset Collection & Generation**
3. **Data Preprocessing**
4. **Model Architecture (CNN + LSTM)**
5. **Model Training**
6. **Model Evaluation**
7. **Result Analysis**
8. **Conclusion**
9. **Future Scope**
10. **Demonstration**


## Stage 1: Problem Definition
Traffic congestion in metropolitan areas causes travel delays, higher carbon emissions, and safety hazards.
- **Problem:** Conventional inductive loop sensors embedded under roads require high maintenance and road cutting.
- **Objective:** Develop an end-to-end system that detects vehicles in camera footage using a Convolutional Neural Network (CNN) and predicts future traffic flow volume for short future intervals (+5, +10, +15 minutes) using an LSTM network.
- **Intended Use:** Intelligent Transportation Systems (ITS), adaptive traffic light controls, and urban congestion alerting.


## Stage 2: Dataset Collection
We utilize two data components:
1. **Road-Camera Frames:** Perspective frames of highway lanes showing moving vehicles across 4 classes: Cars, SUVs, Trucks/Buses, Motorcycles.
2. **Sequential Traffic Time-Series:** 5-minute interval records capturing flow volume (vehicles/5-min), average speed (km/h), and road occupancy (%).


In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("PyTorch Version:", torch.__version__)
print("OpenCV Version:", cv2.__version__)


## Stage 3: Data Preprocessing
- **Vision Data:** Images are resized to 64x64, normalized to [0, 1], and split into 70% Train, 15% Validation, and 15% Test partitions.
- **Time-Series Data:** Scaled using Min-Max normalization and formatted into sliding historical windows of length 12 (past 60 minutes) to predict the next 3 intervals (future 15 minutes).


In [ ]:
# Check existing generated dataset metadata
if os.path.exists("data/vision_metadata.csv") and os.path.exists("data/timeseries.csv"):
    df_vision = pd.read_csv("data/vision_metadata.csv")
    df_series = pd.read_csv("data/timeseries.csv")
    print(f"Loaded {len(df_vision)} vision samples and {len(df_series)} time-series steps.")
    print(df_series.head())


## Stage 4: Model Development
We build two specialized architectures:
1. **SimpleTrafficCNN:** 3 convolutional layers with batch normalization and max-pooling, followed by an adaptive pooling bottleneck that predicts vehicle density and congestion state.
2. **SimpleTrafficLSTM:** 2-layer LSTM taking historical sequences and predicting multi-horizon traffic volume.


In [ ]:
class SimpleTrafficCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_count = nn.Linear(64, 1)
        self.fc_state = nn.Linear(64, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        feat = torch.flatten(self.gap(x), 1)
        pred_count = self.fc_count(feat).squeeze(1)
        pred_state = self.fc_state(feat)
        return pred_count, pred_state

class SimpleTrafficLSTM(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=48, forecast_steps=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.fc_flow = nn.Linear(hidden_dim, forecast_steps)
        self.fc_state = nn.Linear(hidden_dim, 3)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc_flow(last_hidden), self.fc_state(last_hidden)

print("CNN & LSTM architectures defined successfully.")


## Stage 5: Model Training
- **Optimizer:** Adam Optimizer
- **Loss:** Mean Squared Error for flow volume + CrossEntropy for congestion states
- **Epochs:** 6-10 epochs for CPU efficiency


In [ ]:
# Instantiate models
cnn_model = SimpleTrafficCNN()
lstm_model = SimpleTrafficLSTM()

if os.path.exists("models/cnn_simple.pt") and os.path.exists("models/lstm_simple.pt"):
    cnn_model.load_state_dict(torch.load("models/cnn_simple.pt"))
    lstm_model.load_state_dict(torch.load("models/lstm_simple.pt"))
    print("Pre-trained checkpoints loaded successfully from models/.")


## Stage 6: Model Evaluation
We evaluate the models on the test set using:
- **Mean Absolute Error (MAE)** for vehicle count and multi-step flow.
- **Accuracy, Precision, Recall, and F1-Score** for congestion state classification.
- **Confusion Matrix** to inspect classification distribution.


In [ ]:
# Display saved evaluation charts
if os.path.exists("outputs/evaluation_results.png"):
    from PIL import Image
    img = Image.open("outputs/evaluation_results.png")
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Stage 6: Model Evaluation Results (Confusion Matrix & Forecast Curve)")
    plt.show()


## Stage 7: Result Analysis
- **Strengths:** The CNN achieves sub-vehicle density counting accuracy. The LSTM successfully tracks daily morning and evening traffic volume peaks.
- **Error Modes:** Abrupt non-recurrent anomalies (such as accidents) cause brief prediction lag until the sequence buffer registers the speed drop.
- **Limitations:** Single-camera perspectives can experience occlusion when large trucks hide smaller vehicles.

---
## Stage 8: Conclusion
The project met all objectives, proving that video feeds processed with CNNs combined with LSTM sequence models provide reliable short-term traffic flow forecasts without physical road sensors.

---
## Stage 9: Future Scope
1. Implement Graph Convolutional Networks (GCN) for interconnected city-wide intersections.
2. Deploy onto low-power edge computing hardware (e.g. NVIDIA Jetson) using TensorRT.
3. Enhance training with synthetic nighttime and rainy conditions.

---
## Stage 10: Live Demonstration
Testing the end-to-end pipeline with a sample camera frame and past traffic sequence:


In [ ]:
# Live Test Sample
demo_frame = np.ones((64, 64, 3), dtype=np.uint8) * 80
cv2.rectangle(demo_frame, (16, 0), (48, 64), (50, 50, 50), -1)
for (vx, vy) in [(24, 10), (38, 15), (28, 25), (35, 35), (22, 42), (40, 48), (30, 55)]:
    cv2.rectangle(demo_frame, (vx - 2, vy - 2), (vx + 2, vy + 2), (0, 220, 255), -1)

tensor_img = torch.tensor(demo_frame, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0

cnn_model.eval()
with torch.no_grad():
    p_cnt, p_st = cnn_model(tensor_img)
    detected_count = p_cnt.item()
    c_label = ["Free Flow", "Moderate Flow", "Heavy Congestion"][torch.argmax(p_st).item()]

print(f"CNN Detected Count: {detected_count:.1f} vehicles | Traffic Status: {c_label}")

# Display Test Frame
plt.figure(figsize=(3, 3))
plt.imshow(cv2.cvtColor(demo_frame, cv2.COLOR_BGR2RGB))
plt.title(f"CNN Detections: {detected_count:.1f} veh")
plt.axis('off')
plt.show()
